# Shallow neural network (1 hidden layer) for occupancy detection

Same data, split, cross-validation, metrics and baseline as `logistic_regression.ipynb`;
only the model changes, so the results are directly comparable (project step 7).

Run `logistic_regression.ipynb` first: the last cell here compares against its saved results.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, recall_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

## Settings

In [ ]:
DATA_DIR = Path("..") / "data" / "processed"
RESULTS_DIR = Path("..") / "results"
RESOLUTIONS = ["1min_master", "5min", "15min", "30min", "60min"]

TIME_COL = "timestamp"
FEATURES = ["co2_ppm", "rh_pct", "temp_in_c"]
TARGET = "presence"

# Same test period as the logistic regression notebook.
TEST_START = "2020-05-25"

PARAM_GRID = {
    "mlpclassifier__hidden_layer_sizes": [(4,), (8,), (16,), (32,), (64,)],
    "mlpclassifier__activation": ["relu", "tanh"],
    "mlpclassifier__alpha": [1e-4, 1e-2, 1],  # L2 regularization
}


def load(resolution):
    path = DATA_DIR / f"lgh3_may2020_{resolution}.csv"
    return pd.read_csv(path, parse_dates=[TIME_COL]).sort_values(TIME_COL)

## Train and evaluate one resolution
- Chronological split and time-series cross-validation, exactly as for logistic regression.
- `MLPClassifier` has no `class_weight`, so "balanced" sample weights are passed instead
  (the logistic regression chose `class_weight="balanced"` at every resolution).
- Adam optimizer with early stopping: training stops when the score on an internal 10% validation
  slice of the training fold stops improving. `random_state` makes runs reproducible.

In [ ]:
def evaluate(df):
    train = df[df[TIME_COL] < TEST_START]
    test = df[df[TIME_COL] >= TEST_START]
    X_train, y_train = train[FEATURES], train[TARGET].astype(int)
    X_test, y_test = test[FEATURES], test[TARGET].astype(int)

    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(max_iter=500, early_stopping=True, random_state=0),
    )
    search = GridSearchCV(model, PARAM_GRID, cv=TimeSeriesSplit(n_splits=5),
                          scoring="f1_macro", n_jobs=-1)
    search.fit(X_train, y_train,
               mlpclassifier__sample_weight=compute_sample_weight("balanced", y_train))
    y_pred = search.predict(X_test)

    p = search.best_params_
    scores = {
        "train rows": len(train),
        "test rows": len(test),
        "hidden units": p["mlpclassifier__hidden_layer_sizes"][0],
        "activation": p["mlpclassifier__activation"],
        "alpha": p["mlpclassifier__alpha"],
        "CV macro F1": search.best_score_,
        "test macro F1": f1_score(y_test, y_pred, average="macro"),
        "test accuracy": accuracy_score(y_test, y_pred),
        "baseline accuracy": y_test.mean(),  # always predict "occupied"
        "recall empty": recall_score(y_test, y_pred, pos_label=0),
    }
    return search, scores, y_test, y_pred

## Compare time resolutions
Takes a few minutes, mostly for the 1-minute data.

In [ ]:
runs = {}
for res in RESOLUTIONS:
    runs[res] = evaluate(load(res))

results = pd.DataFrame({res: run[1] for res, run in runs.items()}).T
RESULTS_DIR.mkdir(exist_ok=True)
results.to_csv(RESULTS_DIR / "shallow_nn.csv")
results

## Best resolution in detail
Chosen by cross-validation score, not test score.

In [ ]:
best_res = results["CV macro F1"].astype(float).idxmax()
search, scores, y_test, y_pred = runs[best_res]
print(f"Best resolution: {best_res}  ({search.best_params_})\n")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["empty", "occupied"]))

## Shallow NN vs logistic regression (test macro F1)

In [ ]:
lr = pd.read_csv(RESULTS_DIR / "logistic_regression.csv", index_col=0)
pd.DataFrame({
    "logistic regression": lr["test macro F1"],
    "shallow NN": results["test macro F1"].astype(float),
    "baseline accuracy": results["baseline accuracy"].astype(float),
}).round(3)